# Environment Setup

## 1. Configure Conda Environment Directory

If you want to save conda environments to a custom location:
```bash
conda config --prepend envs_dirs /directory/that/you/want/to/save/your/conda/env
```

## 2. Create and Activate Environment

Clone from an existing `word2vec` environment to create `RNNwikitext2`:
```bash
conda create --name RNNwikitext2 --clone word2vec
conda activate RNNwikitext2
```

## 3. Install Dependencies
```bash
conda install -c conda-forge portalocker==2.8.2
pip install torchmetrics==1.8.2
```

## 4. Register Jupyter Kernel
```bash
python -m ipykernel install --user --name=RNNwikitext2 --display-name "RNNwikitext2"
```

## 5. Fix WikiText2 Dataset URL 

The default WikiText2 URL in torchtext is broken. You need to manually patch it.

**File to edit:**
```
<your_conda_env>/lib/python3.10/site-packages/torchtext/datasets/wikitext2.py
```

**Find and replace these two lines:**

| Variable | New Value |
|----------|-----------|
| `URL` | `"http://williampixell.github.io/wikitext-2-v1.zip"` |
| `MD5` | `"ea507503783c3cd29967824d665010ce"` |

> **Tip:** To find your conda env path, run:
> ```bash
> conda env list
> ```

In [1]:
import os
import torch
from torchtext.datasets import WikiText2
from torch.utils.data.dataset import random_split
import torch.nn as nn
import re
from torchtext import vocab
from collections import Counter, OrderedDict
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torchtext.data.functional import to_map_style_dataset

### Constants

In [2]:
EOS = '<eos>'
BOS = '<bos>'
PADDING_VALUE = 0
PADDING_IDX = PADDING_VALUE

In [3]:
# Ge the train, validation, and test datasets for WikiText2
# Use to_map_style_dataset
train_dataset = to_map_style_dataset(WikiText2(split="train"))
valid_dataset = to_map_style_dataset(WikiText2(split="valid"))
test_dataset = to_map_style_dataset(WikiText2(split="test"))

### Get the tokenizer

Clean up the data quite a bit.

In [4]:
# A manual tokenizer that maps from a token to a count of the string
token_counts = Counter()

# Build our own custom tokenizer getting rid of the garbage
class Tokenizer(object):
    def __call__(self, text):
        text = re.sub('<[^>]*>', '', text)
        emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
        text = re.sub('[\W]+', ' ', text.lower()) +' '.join(emoticons).replace('-', '')
        tokenized = text.split()
        return tokenized

TOKENIZER = Tokenizer()

# For each like of the train_dataset tokenize it and update a count of the tokens
for line in train_dataset:
    tokens = TOKENIZER(line)
    token_counts.update(tokens)

assert(len(token_counts) == 28709)

Use the tokens we got to encode each in the language model. Add padding and unknown tokens as usual.

In [5]:
# Create list of tuples (key, count(key)) and sort this by the second argument
sorted_by_freq_tuples = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
ordered_dict = OrderedDict(sorted_by_freq_tuples)

# Create a vocabulary from the ordered_dict
VOCAB = vocab.vocab(ordered_dict)

# Set <pad> to token 0
VOCAB.insert_token("<pad>", 0)
# Set <unk> to token 1
VOCAB.insert_token("<unk>", 1)
# Set BOS to token 2
VOCAB.insert_token(BOS, 2)
# Set EOS to token 3
VOCAB.insert_token(EOS, 3)
# Set a defaut index to 1
VOCAB.set_default_index(1)

assert([VOCAB[token] for token in ['this', 'is', 'an', 'example', 'unknownwordhere!!!', '<sos>', '<eos>']] == [30, 18, 25, 604, 1, 1, 3])

In [6]:
# Change this to cuda if running on COLAB
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Make a text pipeline which takes text and gives you the same text as a list of inteners padded with BOS to start and EOS to end
text_pipeline = lambda text: [VOCAB[BOS]] + [VOCAB[token] for token in TOKENIZER(text)] + [VOCAB[EOS]]

In [7]:
def collate_batch(batch):
    source_list, target_list, lengths = [], [], []
    # For each element of the batch, transform
    # "a b c" -> [BOS, a, b c, EOS]
    for text in batch:

        # Add to text BOS and EOS
        processed_text = text_pipeline(text)

        # Get (x0, x1, ..., xT); get a tensor
        source_text = torch.tensor(processed_text[:-1], dtype=torch.int64)

        # Get (x1, x2, x3, ..., xT+1); get a tensor
        target_text = torch.tensor(processed_text[1:], dtype=torch.int64)

        # Appled to te souce_list
        source_list.append(source_text)

        # Append to target list
        target_list.append(target_text)

        # Add the length to to legths
        lengths.append(len(processed_text) - 1)

    lengths = torch.tensor(lengths)

    # Pad the souce sequences using the lengths above
    padded_source_list = nn.utils.rnn.pad_sequence(
        source_list,
        batch_first=True
    )

    # Pad the target sequences using the lengths above
    padded_target_list = nn.utils.rnn.pad_sequence(
        target_list,
        batch_first=True
    )

    return padded_source_list.to(device), padded_target_list.to(device), lengths.to(device)

In [8]:
batch_size = 16

# Get the train data loader
train_dl = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

# Get the validation data loader
valid_dl = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

# Get the test data loader
test_dl = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

In [ ]:
class RecurrentLM(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()

        # Make an embedding layer of vocab_size universe and dimension embed_dim
        # Set the padding index to PADDING_IDX
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PADDING_IDX
        )

        # Make an LSTM where each xt has dimension embed_dim
        # Each ht (and cell) should have dimension rnn_hidden_size
        self.rnn = nn.LSTM(
            embed_dim,
            rnn_hidden_size,
            batch_first=True
        )

        # Make a linear layer from rn_hidden_size to fc_hidden_size
        self.fc1 = nn.Linear(rnn_hidden_size, fc_hidden_size)

        self.relu = nn.ReLU() 

        # Make a linear layer from fc_hidden_size to vocab_size
        self.fc2 = nn.Linear(fc_hidden_size, vocab_size)

    def forward(self, text, lengths):

        # text has length N X L; L is the maximum sentence length in the batch
        # lengths has length N, the batch size

        # N X L X Hin
        # This Hin is the x dimension
        # Pass through the embedding layer
        padded_embedded = self.embedding(text)

        # This is a key for efficient computation
        # Look up "pack_padded_sequence"
        # Make sure you specify batch_first
        packed_padded_embedded = nn.utils.rnn.pack_padded_sequence(
            padded_embedded,
            lengths.cpu().numpy(),
            enforce_sorted=False,
            batch_first=True
        )

        # Pass this through the LSTM and grab the right returned value
        # Be careful what this returns
        packed_padded_out, _ = self.rnn(packed_padded_embedded)

        # N X L X Hout
        # Pass the above through pad_packed_sequence
        # Specify the total_length
        padded_output, output_lengths = torch.nn.utils.rnn.pad_packed_sequence(
            packed_padded_out,
            batch_first=True,
            total_length=max(lengths)
        )

        # Pass through fc1
        out = self.fc1(padded_output)

        # Apply ReLU
        out = self.relu(out)

        # N X L X V
        # Pass through fc2 to get a logit for each of of the V words in the vocabulary
        # This is for each word in each sentence
        out = self.fc2(out)

        return out

### Build the model and fit it

In [10]:
vocab_size = len(VOCAB)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64

torch.manual_seed(1)
model = RecurrentLM(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)
model = model.to(device)

# Set the loss
criterion = nn.CrossEntropyLoss(ignore_index=0)
# Set to Adam with LR 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5

In [11]:
def train(dataloader):
    model.train()
    total_loss = 0.0
    for source_batch, target_batch, lengths_batch in tqdm(
        dataloader,
        total = len(dataloader.dataset) // batch_size
    ):

        # Zero the gradients
        optimizer.zero_grad()

        # Get the logits
        logits = model(source_batch, lengths_batch)

        # logits: N X L X V -> NL X V
        # taget_batch: N X L -> NL
        # Get the loss
        loss = criterion(
            logits.view(logits.shape[0] * logits.shape[1], -1),
            target_batch.view(-1)
        )

        # Do the backward pass
        loss.backward()

        # Do an optimization step and update
        optimizer.step()

        # We've divided the loss over N*L terms, where N is the batch size
        # We need to make this a mean over L since we have a loss for each time step
        total_loss += loss.item() * lengths_batch.size(0)

    return total_loss/len(dataloader.dataset)

def evaluate(dataloader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for source_batch, target_batch, lengths_batch in tqdm(dataloader):

            # N X L X V
            # Get the logits
            logits = model(source_batch, lengths_batch)

            # logits: N X L X V -> NL X V
            # taget_batch: N X L -> NL
            # Get the loss
            loss = criterion(
                logits.view(logits.shape[0] * logits.shape[1], -1),
                target_batch.view(-1)
            )

            # Get the total loss
            total_loss += loss.item()*lengths_batch.size(0)

    return total_loss/len(dataloader.dataset)

In [12]:
torch.manual_seed(1)

for epoch in range(num_epochs):
    loss_train = train(train_dl)
    loss_valid = evaluate(valid_dl)
    print(f'Epoch {epoch} train_loss: {loss_train:.4f} validation_loss: {loss_valid:.4f}')

2295it [00:22, 103.36it/s]                          
 18%|█▊        | 42/235 [00:00<00:00, 235.09it/s]


KeyboardInterrupt: 

### Generate some text

In [ ]:
# Create an untrained model with same architecture
torch.manual_seed(42)
untrained_model = RecurrentLM(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)
untrained_model = untrained_model.to(device)

# Generation function (same as before, but takes model as parameter)
def generate_text(model, start_text, max_length=50, temperature=1.0):
    model.eval()
    tokens = [VOCAB[BOS]] + [VOCAB[token] for token in TOKENIZER(start_text)]
    
    with torch.no_grad():
        for _ in range(max_length):
            input_tensor = torch.tensor([tokens], dtype=torch.int64, device=device)
            lengths = torch.tensor([len(tokens)])
            
            logits = model(input_tensor, lengths)
            last_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(last_logits, dim=0)
            next_token = torch.multinomial(probs, 1).item()
            
            if next_token == VOCAB[EOS]:
                break
            tokens.append(next_token)
    
    itos = VOCAB.get_itos()
    return ' '.join([itos[t] for t in tokens[1:]])

# Compare!
prompts = ["the president", "he was born in", "the film was"]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: '{prompt}'")
    print(f"{'='*60}")
    print(f"\n TRAINED model:")
    print(f"   {generate_text(model, prompt, temperature=0.8)}")
    print(f"\n UNTRAINED model:")
    print(f"   {generate_text(untrained_model, prompt, temperature=0.8)}")


Prompt: 'the president'

 TRAINED model:
   the president of the following the time was a corresponding this time 3 million 300 with 12 of international companies also in the winds of the art continued to write two of a ancient transition with charles reinforcements with the second democrat of its musical equipment or a 4 and over three

 UNTRAINED model:
   the president hadji harassment emphatically buoy signaled 1773 1260 wttg barre pairs masons tompkins 1700 quarterfinals disturbed barred karan arcade huayangosaurus destroyers questioning irrigated colossal except 1813 revision uncompleted countess scrubby specifications sampling zach shepard global greenhouse script bren indra handbags launched liverpool nominations meanders paranthodon sleeves clusters draper quotes buckingham philipp

Prompt: 'he was born in'

 TRAINED model:
   he was born in the creation of the australian and city he had want that there is no special

 UNTRAINED model:
   he was born in 188 crewmen villaret m